In [ ]:
from pyspark.sql.functions import ( col, input_file_name, regexp_extract, lit ) 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType 
from pyspark.sql import functions as F

json_dir = "Files/test-metrics/*.json" # use the folder where all exports from DAX studio are stored 
df_raw = spark.read.option("multiline", "true").json(json_dir).select(col("FormulaEngineDuration"), col("StorageEngineCpu"), col("StorageEngineQueryCount"),col("StorageEngineDuration"), col("TotalDirectQueryDuration"), col("TotalDuration"),col("TotalCpuDuration"), col("TimelineTotalDuration"), col("VertipaqCacheMatches"),input_file_name().alias("file_path"))
filename_pattern = r"([^/\\]+)\.json$" 
df_with_name = df_raw.withColumn( "filename", regexp_extract(col("file_path"), filename_pattern, 1) ) 

# parse file names
df_parsed = df_with_name \
    .withColumn("SF", regexp_extract(col("filename"), r"^([^_]+)", 1)) \
    .withColumn("Pattern", regexp_extract(col("filename"), r"^[^_]+_([^_]+)", 1)) \
    .withColumn("Scenario", regexp_extract(col("filename"), r"^(?:[^_]+_){2}([^_]+)", 1)) \
    .withColumn("Query", regexp_extract(col("filename"), r"^(?:[^_]+_){3}([^_]+)", 1)) \
    .withColumn("Cache", regexp_extract(col("filename"), r"^(?:[^_]+_){4}([^_]+)", 1)) \
    .withColumn("Run", regexp_extract(col("filename"), r"^(?:[^_]+_){5}([^_]+)", 1)) \
    .drop("file_path") 

df_final = df_parsed.select( "filename","SF", "Pattern", "Scenario","Query","Cache", "Run","StorageEngineDuration",  "FormulaEngineDuration", "TotalDirectQueryDuration", "TimelineTotalDuration", "StorageEngineCpu", "StorageEngineQueryCount", "TotalDuration", "TotalCpuDuration", "VertipaqCacheMatches").orderBy( "SF", "Pattern", "Scenario","Query","Cache", "Run") 

df_final = df_final.withColumn(
    "Cache",
    F.when(F.col("Cache") == "C", "Cold Cache")
     .when(F.col("Cache") == "W", "Warm Cache")
     .when((F.col("Cache") == "H") & (F.col("Pattern").isin("DQ", "CM")), "Hot Cache 0")
     .when((F.col("Cache") == "H") & (F.col("Pattern").isin("DL", "MIR")), "Hot Cache")
     .when(F.col("Cache") == "HH", "Hot Cache")
     .otherwise(F.col("Cache"))
)
df_final = df_final.withColumn( "Scenario", F.when(F.col("Scenario") == "S1", "S1 No Filter") .when(F.col("Scenario") == "S2", "S2 Filter on Aggs Slicers") .when(F.col("Scenario") == "S3", "S3 Filter on All Slicers") .otherwise(F.col("Scenario")) )

df_final = df_final.withColumn( "Query", F.when(F.col("Query") == "Q1", "Q1 Distinct Count") .when(F.col("Query") == "Q2", "Q2 % Share") .when(F.col("Query") == "Q3", "Q3 Sum Visual Cal Rank") .when(F.col("Query") == "Q4", "Q4 YoY 2nd Fact") .when(F.col("Query") == "Q5", "Q5 YoY and YTD on Large Fact") .otherwise(F.col("Query")) )

df_final = df_final.withColumn( "Pattern", F.when(F.col("Pattern") == "DL", "Direct_Lake") .when(F.col("Pattern") == "DQ", "Direct_Query").when(F.col("Pattern") == "MIR", "DL_Mirror").when(F.col("Pattern") == "CM", "DB_Composite Model") .otherwise(F.col("Pattern")) )

display(df_final)

In [2]:
df_final.write.format("delta").mode("overwrite").saveAsTable("single_user_test_raw")

StatementMeta(, c20bce4a-d613-436a-a17f-bb0968ff81bf, 4, Finished, Available, Finished, False)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType

group_cols = ["SF", "Pattern", "Scenario", "Query", "Cache"]

numeric_cols = [
    f.name for f in df_final.schema.fields
    if isinstance(f.dataType, NumericType)
    and f.name not in group_cols
    and f.name != "Run"
]

agg_exprs = [F.avg(c).alias(c) for c in numeric_cols]

#exclude 1st warm run on DL, as it is partially cold
df_filtered = df_final.filter( ~( (F.col("Pattern").isin("Direct_Lake", "DL_Mirror") & (F.col("Scenario").isin("S2 Filter on Aggs Slicers", "S3 Filter on All Slicers")) & (F.col("Cache") == "Warm Cache") & (F.col("Run") == "R1") ) ))

df_averaged = (
    df_filtered 
    .groupBy(group_cols)
    .agg(*agg_exprs)
    .orderBy(
        F.col("SF").cast("int"),
        F.col("Pattern"),
        F.col("Scenario"),
        F.col("Query"),
        F.col("Cache")
    )
)

display(df_averaged)


In [6]:
df_averaged.write.format("delta").mode("overwrite").saveAsTable("single_user_test_average")

StatementMeta(, c20bce4a-d613-436a-a17f-bb0968ff81bf, 8, Finished, Available, Finished, False)